# Cardiac Nexus — ECG Multi-Label Experiment: Diagnostic Superclasses

## Objective

Extend the binary MI/non-MI baseline to a multi-label classifier over PTB-XL's five diagnostic superclasses: NORM (normal), MI (myocardial infarction), STTC (ST/T change), CD (conduction disturbance), and HYP (hypertrophy). A recording can carry more than one label, so this is a multi-label, not multi-class, problem.

> Scope: research evaluation only. This model is a research artifact and must not be used as a standalone basis for medical decisions.

## Experimental procedure

1. Download PTB-XL from PhysioNet (reused from the local cache if the baseline notebook already downloaded it).
2. Build a multi-hot label vector over the five diagnostic superclasses.
3. Use the official patient-level PTB-XL folds: folds 1-8 for training, 9 for validation, and 10 for testing.
4. Load the 100 Hz, 10-second signals and standardize each lead.
5. Train the same 1D convolutional backbone as the baseline, with a 5-way sigmoid output head.
6. Evaluate per-class AUROC/average precision plus macro-averaged summaries, and per-class confusion matrices at a 0.5 threshold.

In [ ]:
!pip -q install wfdb "pandas==2.2.2" numpy scikit-learn matplotlib seaborn torch tqdm

In [ ]:
from pathlib import Path
import ast
import random
import requests
import zipfile

import numpy as np
import pandas as pd
import wfdb
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

SEED = 42
DATA_ROOT = Path("/content/data")
PTBXL_DIR = None
PTBXL_ZIP = DATA_ROOT / "ptb-xl-1.0.3.zip"
PTBXL_URL = "https://physionet.org/static/published-projects/ptb-xl/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3.zip"
SUPERCLASSES = ["NORM", "MI", "STTC", "CD", "HYP"]
BATCH_SIZE = 64
EPOCHS = 15
LEARNING_RATE = 1e-3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)

In [ ]:
# Download the official PTB-XL archive once into the temporary Colab workspace.
DATA_ROOT.mkdir(parents=True, exist_ok=True)
metadata_paths = list(DATA_ROOT.rglob("ptbxl_database.csv"))
if not metadata_paths:
    if PTBXL_ZIP.exists() and not zipfile.is_zipfile(PTBXL_ZIP):
        PTBXL_ZIP.unlink()

    if not PTBXL_ZIP.exists():
        response = requests.get(PTBXL_URL, stream=True, timeout=120)
        response.raise_for_status()
        total_bytes = int(response.headers.get("content-length", 0))
        downloaded_bytes = 0
        with open(PTBXL_ZIP, "wb") as output_file:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    output_file.write(chunk)
                    downloaded_bytes += len(chunk)
                    if downloaded_bytes % (100 * 1024 * 1024) < 1024 * 1024:
                        if total_bytes:
                            print(f"Downloaded {downloaded_bytes / 1024**3:.2f} / {total_bytes / 1024**3:.2f} GB")
                        else:
                            print(f"Downloaded {downloaded_bytes / 1024**3:.2f} GB")

    if not zipfile.is_zipfile(PTBXL_ZIP):
        raise RuntimeError("The PTB-XL download is incomplete or not a ZIP file. Please rerun this cell.")
    with zipfile.ZipFile(PTBXL_ZIP) as archive:
        archive.extractall(DATA_ROOT)
    metadata_paths = list(DATA_ROOT.rglob("ptbxl_database.csv"))

if not metadata_paths:
    raise FileNotFoundError("PTB-XL metadata was not found after extraction.")
PTBXL_DIR = metadata_paths[0].parent

metadata = pd.read_csv(PTBXL_DIR / "ptbxl_database.csv", index_col=0)
scp = pd.read_csv(PTBXL_DIR / "scp_statements.csv", index_col=0)

scp = scp[scp["diagnostic"] == 1]
diagnostic_map = scp["diagnostic_class"].dropna().to_dict()

def diagnostic_classes(raw_codes):
    codes = ast.literal_eval(raw_codes) if isinstance(raw_codes, str) else raw_codes
    return {diagnostic_map[code] for code in codes if code in diagnostic_map}

metadata["diagnostic_classes"] = metadata["scp_codes"].apply(diagnostic_classes)

# Multi-hot label vector over the five PTB-XL diagnostic superclasses.
for cls in SUPERCLASSES:
    metadata[cls] = metadata["diagnostic_classes"].apply(lambda classes, cls=cls: int(cls in classes))

# Drop recordings that have no diagnostic superclass at all (no usable label).
has_any_label = metadata[SUPERCLASSES].sum(axis=1) > 0
metadata = metadata[has_any_label].copy()

print("Total labeled recordings:", len(metadata))
print(metadata[SUPERCLASSES].sum().rename("positive count"))
print("Mean labels per recording:", metadata[SUPERCLASSES].sum(axis=1).mean().round(3))

In [ ]:
# PTB-XL provides patient-wise official folds, reducing train/test leakage.
train_df = metadata[metadata["strat_fold"].between(1, 8)].copy()
val_df = metadata[metadata["strat_fold"] == 9].copy()
test_df = metadata[metadata["strat_fold"] == 10].copy()

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

def load_ecg(row):
    record_path = PTBXL_DIR / row["filename_lr"]
    signal, _ = wfdb.rdsamp(str(record_path))
    signal = signal.astype(np.float32).T  # [12 leads, 1000 samples]
    mean = signal.mean(axis=1, keepdims=True)
    std = signal.std(axis=1, keepdims=True) + 1e-6
    return (signal - mean) / std

class PTBXLDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        labels = row[SUPERCLASSES].to_numpy(dtype=np.float32)
        return torch.tensor(load_ecg(row)), torch.tensor(labels)

train_loader = DataLoader(PTBXLDataset(train_df), batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(PTBXLDataset(val_df), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(PTBXLDataset(test_df), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

sample_x, sample_y = next(iter(train_loader))
print("Batch shape:", sample_x.shape, "Labels shape:", sample_y.shape)

In [ ]:
class ECG1DCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(12, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        return self.classifier(self.features(x).squeeze(-1))

model = ECG1DCNN(num_classes=len(SUPERCLASSES)).to(DEVICE)

# Per-class positive weighting to counter label imbalance (e.g. HYP is rarer than NORM).
positive_counts = train_df[SUPERCLASSES].sum().to_numpy()
negative_counts = len(train_df) - positive_counts
pos_weight = torch.tensor(negative_counts / np.maximum(positive_counts, 1), dtype=torch.float32, device=DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
print(model)
print("Per-class positive weights:", dict(zip(SUPERCLASSES, pos_weight.cpu().numpy().round(2))))

In [ ]:
def run_epoch(loader, training):
    model.train(training)
    losses, labels, probabilities = [], [], []
    for signals, targets in tqdm(loader, leave=False):
        signals, targets = signals.to(DEVICE), targets.to(DEVICE)
        with torch.set_grad_enabled(training):
            logits = model(signals)
            loss = criterion(logits, targets)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        losses.append(loss.item() * len(targets))
        labels.append(targets.detach().cpu().numpy())
        probabilities.append(torch.sigmoid(logits).detach().cpu().numpy())
    return (
        np.sum(losses) / len(loader.dataset),
        np.concatenate(labels, axis=0),
        np.concatenate(probabilities, axis=0),
    )

def macro_auroc(y_true, y_prob):
    scores = []
    for i in range(y_true.shape[1]):
        if len(np.unique(y_true[:, i])) < 2:
            continue
        scores.append(roc_auc_score(y_true[:, i], y_prob[:, i]))
    return float(np.mean(scores)) if scores else float("nan")

history = []
for epoch in range(1, EPOCHS + 1):
    train_loss, _, _ = run_epoch(train_loader, training=True)
    val_loss, val_y, val_p = run_epoch(val_loader, training=False)
    val_macro_auc = macro_auroc(val_y, val_p)
    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "val_macro_auroc": val_macro_auc})
    print(f"Epoch {epoch:02d}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, val_macro_AUROC={val_macro_auc:.4f}")

In [ ]:
history_df = pd.DataFrame(history)
test_loss, test_y, test_p = run_epoch(test_loader, training=False)
test_pred = (test_p >= 0.5).astype(int)

per_class_rows = []
for i, cls in enumerate(SUPERCLASSES):
    y_true, y_prob, y_pred = test_y[:, i], test_p[:, i], test_pred[:, i]
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    per_class_rows.append({
        "class": cls,
        "AUROC": roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else float("nan"),
        "Average precision": average_precision_score(y_true, y_prob),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "Sensitivity": tp / max(tp + fn, 1),
        "Specificity": tn / max(tn + fp, 1),
    })

per_class_df = pd.DataFrame(per_class_rows).set_index("class")
print("Test loss:", test_loss)
print(per_class_df.round(4))
print("\nMacro-averaged AUROC:", per_class_df["AUROC"].mean().round(4))
print("\nFull classification report (subset accuracy is not meaningful for multi-label):")
print(classification_report(test_y, test_pred, target_names=SUPERCLASSES, zero_division=0))

fig, axes = plt.subplots(1, len(SUPERCLASSES), figsize=(4 * len(SUPERCLASSES), 4))
for i, cls in enumerate(SUPERCLASSES):
    cm = confusion_matrix(test_y[:, i], test_pred[:, i], labels=[0, 1])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[i], cbar=False)
    axes[i].set_title(cls)
    axes[i].set_xlabel("Predicted")
    axes[i].set_ylabel("Actual")
plt.tight_layout()

torch.save(
    {"model_state_dict": model.state_dict(), "metrics": per_class_df.to_dict(), "classes": SUPERCLASSES},
    "/content/cardio_nexus_ecg_multilabel.pt",
)
print("Saved research checkpoint to /content/cardio_nexus_ecg_multilabel.pt")

## Interpretation and limitations

Per-class test metrics describe performance on the held-out PTB-XL test fold only. Multi-label subset accuracy is not a meaningful summary metric here; use per-class AUROC, average precision, sensitivity, and specificity, plus the macro-averaged AUROC, when comparing this model to the binary baseline. These results do not establish clinical safety, generalization to Indian patients, or readiness for diagnosis. The dataset version, target definition, random seed, metrics, and limitations should accompany each experiment record.